# 💻 CodeAct 에이전트로 복잡한 문제 해결하기

이번 실습에서는 **CodeAct 패턴**을 활용하여 코드를 직접 실행할 수 있는 에이전트를 구현합니다.

> 📢 **CodeAct란?**
> 
> 단순 Function calling 대신 **Python 코드를 생성하고 실행**하여 복잡한 작업을 수행하는 에이전트입니다.
> - **유연성**: 미리 정의된 도구 없이도 다양한 작업 수행
> - **복잡한 로직**: 반복문, 조건문 등 복잡한 로직 처리
> - **데이터 처리**: 대용량 데이터 분석 및 변환

### 🎯 학습 목표
1. Python REPL Tool을 활용한 코드 실행 에이전트 구현
2. 코드 생성 및 실행 워크플로우 이해하기
3. **스트리밍**으로 단계별 실행 과정 확인하기

## 📋 목차

1. [환경 설정](#1-환경-설정)
2. [Python REPL Tool 구현](#2-python-repl-tool-구현)
3. [CodeAct 에이전트 구성](#3-codeact-에이전트-구성)
4. [스트리밍으로 실행 및 테스트](#4-스트리밍으로-실행-및-테스트)
5. [E2B로 샌드박스에서 CodeAct 에이전트 실행하기](#5-e2b로-샌드박스에서-codeact-에이전트-실행하기)

---
## 1. 환경 설정

In [ ]:
%pip install -qU langchain langgraph langchain-openai python-dotenv e2b

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "your-openai-api-key"

print("✅ 환경 설정 완료!")

---
## 2. Python REPL Tool 구현

Python 코드를 안전하게 실행할 수 있는 도구를 만듭니다.

⚠️ **주의**: 실제 환경에서는 샌드박스 환경에서 실행해야 합니다.

In [ ]:
from langchain_core.tools import tool
import sys
from io import StringIO

@tool
def python_repl(code: str) -> str:
    """
    Python 코드를 실행하고 결과를 반환합니다.
    데이터 분석, 계산, 문자열 처리 등 다양한 작업에 사용하세요.
    
    Args:
        code: 실행할 Python 코드
    
    Returns:
        코드 실행 결과 (print 출력 또는 마지막 표현식의 값)
    """
    old_stdout = sys.stdout
    sys.stdout = mystdout = StringIO()
    
    try:
        exec_globals = {"__builtins__": __builtins__}
        exec(code, exec_globals)
        output = mystdout.getvalue()
        return output if output else "코드가 성공적으로 실행되었습니다."
    except Exception as e:
        return f"오류 발생: {type(e).__name__}: {str(e)}"
    finally:
        sys.stdout = old_stdout

tools = [python_repl]
print("✅ Python REPL Tool 구현 완료!")

---
## 3. CodeAct 에이전트 구성

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode
from typing import Literal

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = """You are a CodeAct agent that solves problems by writing and executing Python code.

Rules:
1. Analyze the user's request carefully
2. Write Python code to solve the problem
3. Use the python_repl tool to execute the code
4. Analyze the results and provide a clear answer
5. Always respond in Korean

When writing code:
- Use print() to show results
- Handle potential errors gracefully
- Write clear, readable code"""

def call_model(state: MessagesState):
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

def should_continue(state: MessagesState) -> Literal["tools", "__end__"]:
    last_message = state["messages"][-1]
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "tools"
    return "__end__"

workflow = StateGraph(MessagesState)
workflow.add_node("call_model", call_model)
workflow.add_node("tools", ToolNode(tools))
workflow.add_edge(START, "call_model")
workflow.add_conditional_edges("call_model", should_continue, {"tools": "tools", "__end__": END})
workflow.add_edge("tools", "call_model")

codeact_agent = workflow.compile()
print("✅ CodeAct 에이전트 컴파일 완료!")

In [ ]:
from IPython.display import Image, display
try:
    display(Image(codeact_agent.get_graph().draw_mermaid_png()))
except:
    print("그래프: START → call_model → [tools/END] → call_model → END")

---
## 4. 스트리밍으로 실행 및 테스트

`stream_mode='updates'`를 사용하면 각 노드가 실행될 때마다 결과를 실시간으로 확인할 수 있습니다.

In [ ]:
# 테스트 1: 수학 계산 (스트리밍)
query1 = "1부터 100까지의 소수를 구하고, 그 합을 계산해줘"

print(f"💬 질문: {query1}")
print("=" * 60)

# stream_mode='updates'로 각 단계별 상태 확인
for chunk in codeact_agent.stream({"messages": [{"role": "user", "content": query1}]}, stream_mode="updates"):
    for node_name, updates in chunk.items():
        print(f"\n📍 [{node_name}] 노드 실행")
        
        if "messages" in updates and updates["messages"]:
            msg = updates["messages"][-1]
            
            # Tool 호출 확인
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"   🔧 Tool 호출: {tc['name']}")
                    code = tc['args'].get('code', '')[:200]
                    print(f"   📝 코드:\n{code}...")
            # Tool 결과
            elif hasattr(msg, 'content') and node_name == 'tools':
                print(f"   📋 Tool 결과: {str(msg.content)[:300]}")
            # 최종 답변
            elif hasattr(msg, 'content') and msg.content:
                print(f"\n🤖 최종 답변:\n{msg.content}")

In [ ]:
# 테스트 2: 데이터 분석 (스트리밍)
query2 = """다음 판매 데이터의 월별 총합과 평균을 계산해줘:
1월: 1200, 2월: 1500, 3월: 1800, 4월: 1100, 5월: 2000, 6월: 1700"""

print(f"💬 질문: {query2}")
print("=" * 60)

for chunk in codeact_agent.stream({"messages": [{"role": "user", "content": query2}]}, stream_mode="updates"):
    for node_name, updates in chunk.items():
        print(f"\n📍 [{node_name}] 노드 실행")
        if "messages" in updates and updates["messages"]:
            msg = updates["messages"][-1]
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"   🔧 Tool: {tc['name']}")
            elif hasattr(msg, 'content') and node_name == 'tools':
                print(f"   📋 결과: {str(msg.content)[:200]}")
            elif hasattr(msg, 'content') and msg.content:
                print(f"\n🤖 최종 답변:\n{msg.content}")

In [ ]:
# 테스트 3: 문자열 처리 (스트리밍)
query3 = "'Hello World LangGraph Agent'에서 각 단어의 길이를 구하고 가장 긴 단어를 찾아줘"

print(f"💬 질문: {query3}")
print("=" * 60)

for chunk in codeact_agent.stream({"messages": [{"role": "user", "content": query3}]}, stream_mode="updates"):
    for node_name, updates in chunk.items():
        print(f"\n📍 [{node_name}] 노드 실행")
        if "messages" in updates and updates["messages"]:
            msg = updates["messages"][-1]
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"   🔧 Tool: {tc['name']}")
            elif hasattr(msg, 'content') and node_name == 'tools':
                print(f"   📋 결과: {str(msg.content)[:200]}")
            elif hasattr(msg, 'content') and msg.content:
                print(f"\n🤖 최종 답변:\n{msg.content}")

---
## 5. E2B로 샌드박스에서 CodeAct 에이전트 실행하기

위에서 구현한 `python_repl`은 **로컬 환경에서 직접 `exec()`를 실행**하므로 보안 위험이 있습니다.

[E2B(Environment to Build)](https://e2b.dev/)를 사용하면 **클라우드의 격리된 샌드박스 VM**에서 코드를 실행할 수 있습니다.

> 📢 **E2B 샌드박스의 특징**
> - 🚀 ~150ms 내에 격리된 VM 생성
> - 🔒 호스트 시스템과 완전히 분리된 실행 환경
> - 📓 내장 Jupyter 서버 → 변수 상태가 셀 간 유지됨
> - 📦 샌드박스 내에서 패키지 설치 가능

또한 LangChain 1.0의 `create_agent`를 사용하면 `StateGraph`를 직접 구성하지 않고도 간결하게 에이전트를 만들 수 있습니다.

In [ ]:
%pip install -qU e2b-code-interpreter

In [ ]:
import os
from e2b_code_interpreter import Sandbox
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# E2B API 키 설정 (https://e2b.dev/docs 에서 무료 발급)
if not os.getenv("E2B_API_KEY"):
    os.environ["E2B_API_KEY"] = "your-e2b-api-key"

# ── 1. E2B 샌드박스 생성 ──
sandbox = Sandbox.create()
print(f"✅ E2B 샌드박스 생성 완료!")

# ── 2. E2B Code Interpreter Tool 정의 ──
@tool
def e2b_code_interpreter(code: str) -> str:
    """
    E2B 클라우드 샌드박스에서 Python 코드를 실행합니다.
    격리된 환경에서 안전하게 코드를 실행하고 결과를 반환합니다.
    matplotlib 등 시각화 라이브러리도 사용 가능합니다.

    Args:
        code: 실행할 Python 코드
    """
    execution = sandbox.run_code(code)

    result_parts = []
    # 표현식 결과값 (예: 마지막 줄이 표현식인 경우)
    if execution.text:
        result_parts.append(execution.text)
    # stdout 출력 (print 결과)
    if execution.logs.stdout:
        stdout = "".join(execution.logs.stdout)
        if stdout.strip():
            result_parts.append(stdout.strip())
    # stderr 출력 (경고 등)
    if execution.logs.stderr:
        stderr = "".join(execution.logs.stderr)
        if stderr.strip():
            result_parts.append(f"[stderr] {stderr.strip()}")
    # 에러 발생 시
    if execution.error:
        result_parts.append(f"Error: {execution.error.name}: {execution.error.value}")

    return "\n".join(result_parts) if result_parts else "코드가 성공적으로 실행되었습니다."

# ── 3. create_agent로 에이전트 구성 ──
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

e2b_agent = create_agent(
    model=llm,
    tools=[e2b_code_interpreter],
    system_prompt=(
        "You are a CodeAct agent that solves problems by writing and executing Python code "
        "in a secure E2B cloud sandbox.\n\n"
        "Rules:\n"
        "1. Analyze the user's request carefully\n"
        "2. Write Python code to solve the problem\n"
        "3. Use the e2b_code_interpreter tool to execute the code\n"
        "4. Analyze the results and provide a clear answer\n"
        "5. Always respond in Korean\n\n"
        "When writing code:\n"
        "- Use print() to show results\n"
        "- Handle potential errors gracefully\n"
        "- You can install packages with: import subprocess; subprocess.run(['pip', 'install', 'package'])"
    ),
)

print("✅ E2B CodeAct 에이전트 구성 완료!")

In [ ]:
# E2B 샌드박스에서 코드 실행 테스트 (스트리밍)
query = "1부터 50까지의 소수를 구하고, 그 개수와 합을 계산해줘"

print(f"💬 질문: {query}")
print("=" * 60)

for chunk in e2b_agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="updates",
):
    for node_name, updates in chunk.items():
        print(f"\n📍 [{node_name}] 노드 실행")

        if "messages" in updates and updates["messages"]:
            msg = updates["messages"][-1]

            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"   🔧 Tool 호출: {tc['name']}")
                    code = tc["args"].get("code", "")[:300]
                    print(f"   📝 코드:\n{code}")
            elif hasattr(msg, "content") and node_name == "tools":
                print(f"   📋 실행 결과: {str(msg.content)[:400]}")
            elif hasattr(msg, "content") and msg.content:
                print(f"\n🤖 최종 답변:\n{msg.content}")

In [ ]:
# ── 로컬 CSV 파일을 E2B 샌드박스에 업로드 후 분석 ──
import base64

# 1) 로컬 파일 → 샌드박스로 업로드
with open("./data/netflix_titles.csv", "rb") as f:
    sandbox.files.write("/home/user/netflix_titles.csv", f)
print("✅ netflix_titles.csv → 샌드박스 업로드 완료!")

# 2) 에이전트에게 분석 요청
query2 = """/home/user/netflix_titles.csv 파일을 pandas로 분석해줘.
1. 연도별 콘텐츠 개수 추이를 막대 차트로 시각화
2. 차트를 /home/user/vis_result.png 로 저장
3. 주요 통계(총 콘텐츠 수, Movie vs TV Show 비율, 가장 많은 연도)를 알려줘"""

print(f"💬 질문: {query2}")
print("=" * 60)

result = e2b_agent.invoke({"messages": [{"role": "user", "content": query2}]})

print("\n🤖 최종 답변:")
print(result["messages"][-1].content)

# 3) 샌드박스에서 결과 차트 다운로드 (바이너리 파일은 base64로 전달)
import os
os.makedirs("./vis_result", exist_ok=True)

dl = sandbox.run_code(
    "import base64; print(base64.b64encode(open('/home/user/vis_result.png','rb').read()).decode())"
)
chart_b64 = "".join(dl.logs.stdout).strip()

with open("./vis_result/netflix_chart.png", "wb") as f:
    f.write(base64.b64decode(chart_b64))
print("\n✅ 차트 다운로드 완료: ./vis_result/netflix_chart.png")

In [ ]:
# 사용이 끝나면 샌드박스를 종료합니다 (비용 절약)
sandbox.kill()
print("✅ E2B 샌드박스가 종료되었습니다.")

---
## 💡 정리

### 스트리밍 모드

| stream_mode | 설명 |
|-------------|------|
| `updates` | 각 노드 실행 결과를 즉시 반환 |
| `messages` | 메시지 단위로 스트리밍 |
| `values` | 전체 상태값 스트리밍 |

### CodeAct vs Function Calling

| 방식 | 장점 | 단점 |
|------|------|------|
| **Function Calling** | 안전, 예측 가능 | 미리 정의된 도구만 사용 |
| **CodeAct** | 유연성, 복잡한 로직 | 보안 주의 필요 |

### 로컬 실행 vs E2B 샌드박스

| 방식 | 장점 | 단점 |
|------|------|------|
| **로컬 `exec()`** | 빠름, 설정 불필요 | 보안 위험, 시스템 접근 가능 |
| **E2B 샌드박스** | 격리된 환경, 안전 | API 키 필요, 네트워크 지연 |

### `StateGraph` vs `create_agent`

| 방식 | 장점 | 단점 |
|------|------|------|
| **StateGraph 직접 구성** | 세밀한 제어, 커스텀 노드 | 보일러플레이트 코드 많음 |
| **create_agent** | 간결, 빠른 프로토타이핑 | 커스터마이징 제한적 |

### 다음 단계
👉 **CH02._05.** Orchestrator-Worker으로 AI 팀 만들어보기